# VER: Discrimination of Persons and Objects in the Ultrasonic Beam

**Project:** Vertically oriented ultrasonic sensor — classify **floor** (0), **box** (1), **human** (2).

**Standalone pipeline:** Load merged data (use `merge_datasets.py` once if needed) → load and inspect data → visualizations → **train all 6 models** → evaluate with **confusion matrix**, **classification report**, and **discriminant difference ΔD** → compare models in a summary table and figures.

---
## 1. Setup

In [2]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    f1_score,
)

# PyTorch (optional — for CNN, TCN, Transformers)
try:
    import torch
    import torch.nn as nn
    from torch.utils.data import TensorDataset, DataLoader
    HAS_TORCH = True
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
except ImportError:
    HAS_TORCH = False
    DEVICE = None

# Project root and paths
ROOT = Path.cwd()
DATASET_DIR = ROOT / "Dataset"
MERGED_PATH = DATASET_DIR / "merged_dataset.csv"
MERGED_SMALL_PATH = DATASET_DIR / "merged_small.csv"
RESULTS_DIR = ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

# Label column index and class names (must match merge script)
LABEL_COL_IDX = 2
CLASS_NAMES = ["floor", "box", "human"]
RANDOM_STATE = 42
TEST_SIZE = 0.2
N_EPOCHS_DL = 15  # epochs for deep learning models

plt.rcParams["figure.dpi"] = 120
plt.style.use("ggplot")
print("Setup OK. Root:", ROOT)
print("PyTorch available:", HAS_TORCH)

ModuleNotFoundError: No module named 'seaborn'

---
## 2. Load Dataset

Use **`Dataset/merged_dataset.csv`** (or **`Dataset/merged_small.csv`** if present). If the merged file does not exist, run once: **`python merge_datasets.py -o Dataset/merged_dataset.csv`**.

The full merged file has ~25k columns; loading it entirely can exhaust memory. If **`merged_small.csv`** exists we use it; otherwise we build it from the merged file (subset of rows, every Nth feature).

In [ ]:
MAX_ROWS_SMALL = 5000
FEATURE_STEP_SMALL = 50
N_COLS_FULL = 25017

if not MERGED_PATH.exists():
    raise FileNotFoundError(
        "Merged file not found. Run once: python merge_datasets.py -o Dataset/merged_dataset.csv"
    )

if MERGED_SMALL_PATH.exists():
    print("Small merged file exists. Using it for loading.")
    DATA_PATH = MERGED_SMALL_PATH
    N_COLS = len(pd.read_csv(MERGED_SMALL_PATH, nrows=0, header=None).columns)
    FEATURE_STEP = 1
else:
    import random
    random.seed(RANDOM_STATE)
    with open(MERGED_PATH, "r", encoding="utf-8") as f:
        total_lines = sum(1 for _ in f)
    keep = set(random.sample(range(total_lines), min(MAX_ROWS_SMALL, total_lines)))
    feat_idx = [0, 1, 2] + list(range(3, N_COLS_FULL))[::FEATURE_STEP_SMALL]
    written = 0
    with open(MERGED_PATH, "r", encoding="utf-8") as fin:
        with open(MERGED_SMALL_PATH, "w", encoding="utf-8", newline="") as fout:
            for idx, line in enumerate(fin):
                if idx not in keep:
                    continue
                parts = line.strip().split(",")
                if len(parts) < N_COLS_FULL:
                    continue
                fout.write(",".join(parts[i] for i in feat_idx) + "\n")
                written += 1
    print(f"Created {MERGED_SMALL_PATH.name}: {written} rows, {len(feat_idx)} columns.")
    DATA_PATH = MERGED_SMALL_PATH
    N_COLS = len(feat_idx)
    FEATURE_STEP = 1

print(f"Data path: {DATA_PATH}")

---
## 3. Load and Display Dataset

In [ ]:
chunk_size = 2000
chunks = []
y_list = []
for chunk in pd.read_csv(DATA_PATH, header=None, chunksize=chunk_size, low_memory=False):
    y_list.append(chunk.iloc[:, LABEL_COL_IDX].values)
    feat_cols = [i for i in range(chunk.shape[1]) if i != LABEL_COL_IDX]
    chunks.append(chunk.iloc[:, feat_cols].astype(np.float32))

X = np.vstack(chunks)
y = np.concatenate(y_list)

print("Shape:", X.shape, "(samples, features)")
print("Labels:", y.shape)
print("Classes:", np.unique(y))
print("Label mapping: 0=floor, 1=box, 2=human")

In [ ]:
df_display = pd.DataFrame({
    "class": [CLASS_NAMES[int(c)] for c in np.unique(y)],
    "label": np.unique(y),
    "count": [np.sum(y == c) for c in np.unique(y)],
})
df_display["pct"] = (df_display["count"] / len(y) * 100).round(1).astype(str) + "%"
df_display

---
## 4. Visualizations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Class distribution
counts = [np.sum(y == c) for c in range(len(CLASS_NAMES))]
axes[0].bar(CLASS_NAMES, counts, color=["#2ecc71", "#3498db", "#e74c3c"])
axes[0].set_title("Class distribution")
axes[0].set_ylabel("Count")
for i, v in enumerate(counts):
    axes[0].text(i, v + max(counts) * 0.01, str(v), ha="center", fontsize=10)

# Feature statistics (mean per feature, then histogram of means)
mean_per_feat = np.mean(X, axis=0)
axes[1].hist(mean_per_feat, bins=50, edgecolor="black", alpha=0.7)
axes[1].set_title("Distribution of feature means")
axes[1].set_xlabel("Mean value")
axes[1].set_ylabel("Count")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "data_overview.png", bbox_inches="tight")
plt.show()

In [ ]:
# Mean signal (across features) per class — first 500 features as a proxy for "signal shape"
n_show = min(500, X.shape[1])
fig, ax = plt.subplots(figsize=(10, 4))
for c, name in enumerate(CLASS_NAMES):
    mask = y == c
    if mask.sum() == 0:
        continue
    mean_signal = X[mask, :n_show].mean(axis=0)
    ax.plot(mean_signal, label=name, alpha=0.8)
ax.set_title("Mean signal (first {} features) by class".format(n_show))
ax.set_xlabel("Feature index")
ax.set_ylabel("Mean value")
ax.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / "mean_signal_by_class.png", bbox_inches="tight")
plt.show()

---
## 5. Preprocessing

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
print("Train:", X_train.shape[0], "Test:", X_test.shape[0])

---
## 6. Training: All Models

Train **SVM** and **Random Forest** first; then (if PyTorch is available) **CNN**, **TCN**, **Conv-Transformer Hybrid**, and **Time Series Transformer**. Predictions are collected for comparison in the next section.

In [ ]:
svm = SVC(kernel="rbf", C=1.0, gamma="scale", random_state=RANDOM_STATE)
svm.fit(X_train, y_train)
y_pred_svm = svm.predict(X_test)

rf = RandomForestClassifier(n_estimators=100, max_depth=20, random_state=RANDOM_STATE)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

# Collect results: (model_name, y_pred) for later comparison
results_list = [("SVM", y_pred_svm), ("Random Forest", y_pred_rf)]
print("SVM and Random Forest trained.")

In [ ]:
# --- Deep learning models (run only if PyTorch is available) ---
if HAS_TORCH:
    seq_len = X_train.shape[1]
    n_classes = len(CLASS_NAMES)

    class CNN1D(nn.Module):
        def __init__(self, input_len, num_classes=3, channels=64):
            super().__init__()
            self.conv = nn.Sequential(
                nn.Conv1d(1, channels, 7, padding=3),
                nn.BatchNorm1d(channels), nn.ReLU(), nn.MaxPool1d(2),
                nn.Conv1d(channels, channels * 2, 5, padding=2),
                nn.BatchNorm1d(channels * 2), nn.ReLU(), nn.MaxPool1d(2),
                nn.Conv1d(channels * 2, channels * 2, 3, padding=1),
                nn.BatchNorm1d(channels * 2), nn.ReLU(),
                nn.AdaptiveAvgPool1d(1),
            )
            self.fc = nn.Linear(channels * 2, num_classes)
        def forward(self, x):
            x = self.conv(x).flatten(1)
            return self.fc(x)

    class TCNBlock(nn.Module):
        def __init__(self, in_c, out_c, k):
            super().__init__()
            self.conv1 = nn.Conv1d(in_c, out_c, k, padding=(k - 1) // 2)
            self.conv2 = nn.Conv1d(out_c, out_c, k, padding=(k - 1) // 2)
            self.norm1, self.norm2 = nn.BatchNorm1d(out_c), nn.BatchNorm1d(out_c)
            self.down = nn.Conv1d(in_c, out_c, 1) if in_c != out_c else nn.Identity()
        def forward(self, x):
            res = self.down(x)
            x = torch.relu(self.norm1(self.conv1(x)))
            return torch.relu(res + self.norm2(self.conv2(x)))

    class TCN(nn.Module):
        def __init__(self, input_len, num_classes=3, channels=64, levels=4):
            super().__init__()
            layers = []
            in_c = 1
            for _ in range(levels):
                layers.append(TCNBlock(in_c, channels, 7))
                in_c = channels
            self.tcn = nn.Sequential(*layers)
            self.pool = nn.AdaptiveAvgPool1d(1)
            self.fc = nn.Linear(channels, num_classes)
        def forward(self, x):
            return self.fc(self.pool(self.tcn(x)).flatten(1))

    class ConvTransformerHybrid(nn.Module):
        def __init__(self, input_len, num_classes=3, d_model=64, nhead=4, num_layers=2, max_len=512):
            super().__init__()
            self.proj = nn.Conv1d(1, d_model, 7, stride=4, padding=3)
            self.seq_len = (input_len + 2 * 3 - 7) // 4 + 1
            self.pos = nn.Parameter(torch.randn(1, min(self.seq_len, max_len), d_model) * 0.02)
            enc = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward=d_model * 4, dropout=0.1, batch_first=True)
            self.transformer = nn.TransformerEncoder(enc, num_layers=num_layers)
            self.fc = nn.Linear(d_model, num_classes)
        def forward(self, x):
            x = self.proj(x).transpose(1, 2)
            if x.size(1) > self.pos.size(1):
                x = x[:, : self.pos.size(1)]
            elif x.size(1) < self.pos.size(1):
                x = nn.functional.pad(x, (0, 0, 0, self.pos.size(1) - x.size(1)))
            x = x + self.pos[:, : x.size(1)]
            return self.fc(self.transformer(x).mean(dim=1))

    class TimeSeriesTransformer(nn.Module):
        def __init__(self, input_len, num_classes=3, d_model=64, nhead=4, num_layers=2, patch_size=64):
            super().__init__()
            self.patch_size = patch_size
            self.n_patches = (input_len + patch_size - 1) // patch_size
            self.proj = nn.Linear(patch_size, d_model)
            self.pos = nn.Parameter(torch.randn(1, self.n_patches, d_model) * 0.02)
            enc = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward=d_model * 4, dropout=0.1, batch_first=True)
            self.transformer = nn.TransformerEncoder(enc, num_layers=num_layers)
            self.fc = nn.Linear(d_model, num_classes)
        def forward(self, x):
            B, C, L = x.shape
            x = x.squeeze(1)
            pads = (self.patch_size - L % self.patch_size) % self.patch_size
            if pads > 0:
                x = nn.functional.pad(x, (0, pads))
            x = x.view(B, -1, self.patch_size)
            x = self.proj(x) + self.pos
            return self.fc(self.transformer(x).mean(dim=1))

    def train_dl(model, train_loader, n_epochs):
        model.train()
        opt = torch.optim.Adam(model.parameters(), lr=1e-3)
        for epoch in range(n_epochs):
            for xs, ys in train_loader:
                xs, ys = xs.to(DEVICE), ys.to(DEVICE)
                opt.zero_grad()
                loss = nn.functional.cross_entropy(model(xs), ys)
                loss.backward()
                opt.step()
        return model

    def predict_dl(model, X_np, batch_size=256):
        model.eval()
        preds = []
        with torch.no_grad():
            for i in range(0, len(X_np), batch_size):
                batch = torch.from_numpy(X_np[i:i + batch_size]).float().to(DEVICE)
                preds.append(model(batch).argmax(dim=1).cpu().numpy())
        return np.concatenate(preds)

    X_train_dl = torch.from_numpy(X_train).float().unsqueeze(1)
    X_test_dl_np = torch.from_numpy(X_test).float().unsqueeze(1).numpy()
    train_ds = TensorDataset(X_train_dl, torch.from_numpy(y_train).long())
    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=0)

    for name, model_class, kwargs in [
        ("CNN", CNN1D, {}),
        ("TCN", TCN, {}),
        ("Conv-Transformer Hybrid", ConvTransformerHybrid, {"max_len": 512}),
        ("Time Series Transformer", TimeSeriesTransformer, {"patch_size": max(8, min(64, seq_len // 4))}),
    ]:
        print(f"Training {name}...")
        m = model_class(seq_len, num_classes=n_classes, **kwargs).to(DEVICE)
        train_dl(m, train_loader, N_EPOCHS_DL)
        yp = predict_dl(m, X_test_dl_np)
        results_list.append((name, yp))
    print("All deep learning models trained.")
else:
    print("PyTorch not installed. Only SVM and Random Forest were trained.")

---
## 7. Evaluation and Comparison

For each model we report: **confusion matrix**, **classification report** (precision, recall, F1), and **discriminant difference** ΔD = (D₁ − D₂)² / (D₁ + D₂)² (pairwise). Then we compare all models in a summary table.

In [ ]:
def discriminant_difference(cm, i, j):
    d1, d2 = cm[i, i], cm[j, j]
    s = d1 + d2
    return 0.0 if s == 0 else ((d1 - d2) ** 2) / (s ** 2)

def plot_confusion_matrix(y_true, y_pred, title, ax):
    cm = confusion_matrix(y_true, y_pred)
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(CLASS_NAMES)))
    ax.set_yticks(range(len(CLASS_NAMES)))
    ax.set_xticklabels(CLASS_NAMES)
    ax.set_yticklabels(CLASS_NAMES)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center")
    plt.colorbar(im, ax=ax)
    return cm

# Store metrics for comparison
all_cms = {}

In [ ]:
n_models = len(results_list)
n_cols = 3
n_rows = (n_models + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows))
if n_models == 1:
    axes = np.array([[axes]])
elif axes.ndim == 1:
    axes = axes.reshape(1, -1)
for idx, (name, y_pred) in enumerate(results_list):
    r, c = idx // n_cols, idx % n_cols
    ax = axes[r, c]
    cm = plot_confusion_matrix(y_test, y_pred, f"{name} (acc={accuracy_score(y_test, y_pred):.3f})", ax)
    all_cms[name] = cm
for idx in range(n_models, n_rows * n_cols):
    r, c = idx // n_cols, idx % n_cols
    axes[r, c].set_visible(False)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "confusion_all_models.png", bbox_inches="tight")
plt.show()

In [ ]:
for name, y_pred in results_list:
    cm = confusion_matrix(y_test, y_pred)
    print("=" * 50)
    print(name)
    print("=" * 50)
    print(classification_report(y_test, y_pred, target_names=CLASS_NAMES, zero_division=0))
    delta_d = {f"{CLASS_NAMES[i]}-{CLASS_NAMES[j]}": discriminant_difference(cm, i, j)
               for i, j in [(0, 1), (0, 2), (1, 2)]}
    print("Discriminant difference (pairwise):", delta_d)
    print()

---
## 8. Model Comparison Summary

In [ ]:
summary_data = []
for name, y_pred in results_list:
    acc = accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)
    summary_data.append({"Model": name, "Accuracy": acc, "Macro F1": macro_f1})
summary_df = pd.DataFrame(summary_data)
summary_df["Accuracy"] = summary_df["Accuracy"].round(4)
summary_df["Macro F1"] = summary_df["Macro F1"].round(4)
summary_df

In [ ]:
# Bar chart: accuracy and macro F1 per model
fig, ax = plt.subplots(figsize=(max(8, len(results_list) * 1.5), 4))
x = np.arange(len(summary_df))
w = 0.35
ax.bar(x - w/2, summary_df["Accuracy"], w, label="Accuracy")
ax.bar(x + w/2, summary_df["Macro F1"], w, label="Macro F1")
ax.set_xticks(x)
ax.set_xticklabels(summary_df["Model"], rotation=45, ha="right")
ax.set_ylabel("Score")
ax.legend()
ax.set_title("Model comparison (professor metrics)")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "comparison_all_models.png", bbox_inches="tight")
plt.show()

best_acc = summary_df.loc[summary_df["Accuracy"].idxmax(), "Model"]
best_f1 = summary_df.loc[summary_df["Macro F1"].idxmax(), "Model"]
print("Pipeline complete. All models trained and evaluated.")
print(f"Best accuracy: {best_acc}")
print(f"Best macro F1: {best_f1}")
print("Results and figures saved under results/.")